# 13 · Combinaciones rápidas — A1 + variantes

Tres experimentos en ~2 horas usando A1 (oversampling ×2 p50) como base,
que ya demostró ser el mejor config manual en la notebook 11.

| Exp | Descripción | Submissions | Tiempo est. |
|-----|-------------|-------------|-------------|
| A | A1 + 4 meses train final | `submissions10/` | ~30 min |
| B | A1 + ventana 4 meses en walk-forward | `submissions11/` | ~30 min |
| C | Stacking: 2 modelos params distintos + promedio ponderado | `submissions12/` | ~50 min |

**Input:** `dataset_fe_v4.parquet` + params del `mejores_params_optuna_v5.json`  
**Proyecto:** Predicción de Churn Bancario · Data Mining en EyF  
**Deadline:** 11/10/2026 23:59:59

## 0 · Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install lightgbm --quiet

In [ ]:
import gc
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110

RUTA_BASE = Path('/content/drive/MyDrive/Maestría Data Mining/DM en EyF')
RUTA_COMP = RUTA_BASE / 'Competencia 1'

RUTA_SUBS_A = RUTA_COMP / 'submissions10'  # A1 + 4 meses train final
RUTA_SUBS_B = RUTA_COMP / 'submissions11'  # A1 + ventana 4 meses WF
RUTA_SUBS_C = RUTA_COMP / 'submissions12'  # stacking
for r in [RUTA_SUBS_A, RUTA_SUBS_B, RUTA_SUBS_C]:
    r.mkdir(parents=True, exist_ok=True)

CKPT_FE_V4       = RUTA_COMP / 'dataset_fe_v4.parquet'
CKPT_V5          = RUTA_COMP / 'mejores_params_optuna_v5.json'
CKPT_PROBS_13A   = RUTA_COMP / 'probs_v13a.npy'
CKPT_IDS_13A     = RUTA_COMP / 'ids_v13a.npy'
CKPT_PROBS_13B   = RUTA_COMP / 'probs_v13b.npy'
CKPT_IDS_13B     = RUTA_COMP / 'ids_v13b.npy'
CKPT_PROBS_13C   = RUTA_COMP / 'probs_v13c.npy'
CKPT_IDS_13C     = RUTA_COMP / 'ids_v13c.npy'
ARCHIVO_METRICAS = RUTA_COMP / 'metricas_experimentos.csv'

COL_ID      = 'numero_de_cliente'
COL_PERIODO = 'foto_mes'
TARGET_RAW  = 'clase_ternaria'
TARGET      = 'es_baja2'

MESES_CON_TARGET = [202103, 202104, 202105, 202106]
MES_PREDECIR     = 202108
GANANCIA_TP      =  1_072_500
COSTO_FP         =    -27_500
SEED             = 42
SEMILLAS         = [42, 123, 777, 2024]

# A1 config ganadora
OVER_FACTOR = 2
OVER_PCT    = 50
VARS_PERFIL = ['cliente_antiguedad', 'ctrx_quarter', 'mcuentas_saldo']

# Referencia v5 (mejor submit público: 87.07M)
G_V5_F4_REF = 430_980_000  # ganancia fold_4 del v5

def mem_mb(df): return df.memory_usage(deep=True).sum() / 1024**2

print('Checkpoints:')
for ck, n in [(CKPT_FE_V4,'FE v4'),(CKPT_V5,'params v5')]:
    print(f'  {n}: {"✓" if ck.exists() else "✗"}')
print('Submissions: 10/, 11/, 12/')

Checkpoints:
  FE v4: ✓
  params v5: ✓
Submissions: 10/, 11/, 12/


## 1 · Funciones base

In [ ]:
def ganancia_por_threshold(y_true, y_prob, n=500):
    thresholds = np.linspace(0.001, 0.999, n)
    rows = []
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        tp   = ((y_true == 1) & (pred == 1)).sum()
        fp   = ((y_true == 0) & (pred == 1)).sum()
        rows.append({'threshold': t, 'ganancia': tp * GANANCIA_TP + fp * COSTO_FP,
                     'tp': int(tp), 'fp': int(fp), 'n_intervenidos': int(tp + fp)})
    return pd.DataFrame(rows)

def threshold_optimo(y_true, y_prob):
    df_t = ganancia_por_threshold(y_true, y_prob)
    return df_t.loc[df_t['ganancia'].idxmax()].to_dict()

def evaluar_fold(y_true, y_prob, label=''):
    auc = roc_auc_score(y_true, y_prob)
    opt = threshold_optimo(y_true, y_prob)
    print(f'  [{label}] AUC={auc:.4f} | Ganancia=${opt["ganancia"]:,.0f} | '
          f'N={opt["n_intervenidos"]:,} | TP={opt["tp"]} | thresh={opt["threshold"]:.4f}')
    return {'auc': round(auc, 4), **opt}

def guardar_metricas(rows):
    df_n = pd.DataFrame(rows)
    df_n['timestamp'] = datetime.now().isoformat()
    if ARCHIVO_METRICAS.exists():
        hist = pd.read_csv(ARCHIVO_METRICAS)
        pd.concat([hist, df_n], ignore_index=True).to_csv(ARCHIVO_METRICAS, index=False)
    else:
        df_n.to_csv(ARCHIVO_METRICAS, index=False)

def aplicar_oversampling(X_train, y_train, factor=2, pct=50, verbose=False):
    eps      = 1e-6
    vars_ok  = [v for v in VARS_PERFIL if v in X_train.columns]
    if not vars_ok: return X_train.copy(), y_train.copy()
    mask_b2  = y_train == 1
    X_b2     = X_train[mask_b2]
    umbrales = {v: X_b2[v].quantile(pct / 100) for v in vars_ok}
    n_supera = sum((X_b2[v] >= umbrales[v]).astype(int) for v in vars_ok)
    mask_fn  = n_supera >= min(2, len(vars_ok))
    X_fn     = X_b2[mask_fn]
    y_fn     = y_train[mask_b2][mask_fn]
    if len(X_fn) == 0: return X_train.copy(), y_train.copy()
    X_extra  = pd.concat([X_fn] * (factor - 1), ignore_index=True)
    y_extra  = pd.concat([y_fn] * (factor - 1), ignore_index=True)
    if verbose:
        print(f'    Oversampling ×{factor} p{pct}: {len(X_fn):,} baja2 duplicados')
        new_r = (y_train==0).sum()/(y_train==1).sum()+len(y_extra)
        print(f'    Train: {len(X_train):,} → {len(X_train)+len(X_extra):,}')
    return (pd.concat([X_train, X_extra], ignore_index=True),
            pd.concat([y_train, y_extra], ignore_index=True))

def semillerio_y_submission(params, X_tf, y_tf, X_pred, ids_pred,
                             n_opt, ruta_subs, prefijo,
                             ckpt_probs, ckpt_ids,
                             usar_oversampling=True, es_dart=False):
    paso     = max(100, int(n_opt * 0.025))
    cortes   = [max(8000, min(15000, n_opt + i * paso)) for i in range(-2, 3)]
    ts       = datetime.now().strftime('%m%d_%H%M')

    # Checkpoint
    if ckpt_probs.exists() and ckpt_ids.exists():
        probs = list(np.load(ckpt_probs))
        ids_ck = np.load(ckpt_ids)
        n_ya  = len(probs) if np.array_equal(ids_ck, ids_pred) else 0
        if n_ya == 0: probs = []
        else: print(f'  Checkpoint: {n_ya}/{len(SEMILLAS)} semillas')
    else:
        probs = []; n_ya = 0

    if usar_oversampling:
        X_tf_aug, y_tf_aug = aplicar_oversampling(X_tf, y_tf,
            factor=OVER_FACTOR, pct=OVER_PCT, verbose=(n_ya==0))
    else:
        X_tf_aug, y_tf_aug = X_tf, y_tf

    for i_s, sem in enumerate(SEMILLAS[n_ya:], start=n_ya):
        print(f'  Semilla {sem} ({i_s+1}/{len(SEMILLAS)})...', end=' ')
        m = lgb.LGBMClassifier(**{**params, 'random_state': sem})
        if es_dart:
            m.fit(X_tf_aug, y_tf_aug, callbacks=[lgb.log_evaluation(-1)])
        else:
            m.fit(X_tf_aug, y_tf_aug, callbacks=[lgb.log_evaluation(-1)])
        probs.append(m.predict_proba(X_pred)[:, 1])
        del m; gc.collect()
        np.save(ckpt_probs, np.array(probs))
        np.save(ckpt_ids, ids_pred)
        print('✓')

    # Generar CSVs
    archivos = []
    for i_s, sem in enumerate(SEMILLAS):
        for n_corte in cortes:
            ranking = np.argsort(probs[i_s])[::-1][:n_corte]
            ids_int = ids_pred[ranking].astype(int)
            nombre  = f'{prefijo}_sem{sem}_top{n_corte}_{ts}'
            pd.Series(ids_int).to_csv(ruta_subs / f'{nombre}.csv',
                                      index=False, header=False)
            archivos.append(nombre)
    pd.DataFrame({'nombre': archivos}).to_csv(ruta_subs / f'log_{ts}.csv', index=False)
    tiene_e = pd.read_csv(ruta_subs / f'{archivos[0]}.csv', header=None)[0]                .astype(str).str.lower().str.contains('e').any()
    print(f'  {len(archivos)} archivos | sin notación científica: {"✓" if not tiene_e else "✗"}')
    print(f'  Cortes: {cortes}')
    print(f'  → submit {prefijo}_{ts} | {ruta_subs.name}/')
    return probs, cortes, ts

print('✓ Funciones listas.')

✓ Funciones listas.


## 2 · Carga de datos y params

In [ ]:
print('Cargando dataset_fe_v4.parquet...')
df = pd.read_parquet(CKPT_FE_V4)
df = df.sort_values([COL_ID, COL_PERIODO]).reset_index(drop=True)
print(f'Shape: {df.shape[0]:,} × {df.shape[1]} — {mem_mb(df):.1f} MB')

EXCLUIR      = {COL_ID, COL_PERIODO, TARGET_RAW, TARGET, 'es_baja'}
FEATURE_COLS = [c for c in df.columns if c not in EXCLUIR]
print(f'Features: {len(FEATURE_COLS):,}')

mask_e        = df[COL_PERIODO].isin(MESES_CON_TARGET) & df[TARGET].notna()
y_ref         = df.loc[mask_e, TARGET]
SCALE_NATURAL = float((y_ref == 0).sum() / (y_ref == 1).sum())
del y_ref; gc.collect()

with open(CKPT_V5) as f:
    p_v5 = json.load(f)

PARAMS_V5 = {k: v for k, v in p_v5.items()
             if k not in {'ganancia_optuna_fold4','n_trials','n_trials_completados',
                          'fecha','experimento','estado','n_meses_train','usar_is_unbalance'}}
PARAMS_V5.update({'objective':'binary','metric':'binary_logloss',
                  'verbosity':-1,'n_jobs':-1,'random_state':SEED})
if p_v5.get('usar_is_unbalance', False):
    PARAMS_V5['is_unbalance'] = True
    PARAMS_V5.pop('scale_pos_weight', None)
else:
    PARAMS_V5['scale_pos_weight'] = p_v5.get('scale_pos_weight', SCALE_NATURAL * 2)
    PARAMS_V5.pop('is_unbalance', None)

N_MESES_V5 = p_v5.get('n_meses_train', 3)
ES_DART    = PARAMS_V5.get('boosting_type') == 'dart'

# Params alternativos para el stacking (exp C)
# Variante: más hojas y menos regularización para capturar patrones distintos
PARAMS_V5_ALT = {**PARAMS_V5,
    'num_leaves':    min(int(PARAMS_V5.get('num_leaves', 64) * 1.5), 255),
    'learning_rate': PARAMS_V5.get('learning_rate', 0.05) * 0.7,
    'feature_fraction': max(PARAMS_V5.get('feature_fraction', 0.8) - 0.15, 0.4),
    'random_state':  SEED + 100,
}

print(f'Params v5: boosting={PARAMS_V5["boosting_type"]} | ventana={N_MESES_V5}')
print(f'Params alt: num_leaves={PARAMS_V5_ALT["num_leaves"]} | '
      f'lr={PARAMS_V5_ALT["learning_rate"]:.4f} | '
      f'feat_frac={PARAMS_V5_ALT["feature_fraction"]:.2f}')

FOLDS = [
    {'nombre':'fold_2','train':[202103],                'test':202104},
    {'nombre':'fold_3','train':[202103,202104],         'test':202105},
    {'nombre':'fold_4','train':[202103,202104,202105],  'test':202106},
]
FOLDS_4M = [
    {'nombre':'fold_2','train':[202103],                        'test':202104},
    {'nombre':'fold_3','train':[202103,202104],                 'test':202105},
    {'nombre':'fold_4','train':[202103,202104,202105,202106],   'test':202106},
]

def get_meses_train(fold_nombre, n_ventana):
    m = {'fold_2':[202103],'fold_3':[202103,202104],'fold_4':[202103,202104,202105]}
    return m[fold_nombre][-n_ventana:]

def get_split(meses_train, mes_test):
    mt = df[COL_PERIODO].isin(meses_train) & df[TARGET].notna()
    me = (df[COL_PERIODO] == mes_test)      & df[TARGET].notna()
    return (df.loc[mt, FEATURE_COLS].copy(), df.loc[mt, TARGET].astype(int).copy(),
            df.loc[me, FEATURE_COLS].copy(), df.loc[me, TARGET].astype(int).copy())

def fit_modelo(params, Xtr, ytr, Xte, yte, es_dart):
    m = lgb.LGBMClassifier(**params)
    if es_dart:
        m.fit(Xtr, ytr, callbacks=[lgb.log_evaluation(-1)])
    else:
        m.fit(Xtr, ytr, eval_set=[(Xte, yte)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    return m

Cargando dataset_fe_v4.parquet...
Shape: 983,061 × 409 — 1504.7 MB
Features: 405
Params v5: boosting=dart | ventana=3
Params alt: num_leaves=133 | lr=0.0310 | feat_frac=0.40


---
## 3 · Experimento A — A1 + train final con 4 meses

Walk-forward igual que v5 (ventana 3 meses) + oversampling A1.
**Cambio único:** el modelo final (semillerio) entrena con los 4 meses completos
+ oversampling. Más datos en producción.

In [ ]:
print('\n' + '='*60)
print('EXP A — A1 + 4 MESES TRAIN FINAL (submissions10/)')
print('='*60)

rows_a = []
for fold in FOLDS:
    meses_tr = get_meses_train(fold['nombre'], N_MESES_V5)
    Xtr, ytr, Xte, yte = get_split(meses_tr, fold['test'])
    Xtr_aug, ytr_aug = aplicar_oversampling(Xtr, ytr, verbose=(fold['nombre']=='fold_4'))
    m = fit_modelo(PARAMS_V5, Xtr_aug, ytr_aug, Xte, yte, ES_DART)
    prob = m.predict_proba(Xte)[:, 1]
    res  = evaluar_fold(yte.values, prob, f'A {fold["nombre"]}')
    rows_a.append({'fold': fold['nombre'], 'experimento': 'v13a_a1_4m', **res})
    del Xtr, ytr, Xtr_aug, ytr_aug, Xte, yte, m, prob; gc.collect()

guardar_metricas(rows_a)
G_A_F4  = next(r for r in rows_a if r['fold']=='fold_4')['ganancia']
N_OPT_A = int(next(r for r in rows_a if r['fold']=='fold_4')['n_intervenidos'])
print(f'\nfold_4: ${G_A_F4:,.0f} | vs v5: {"+ " if G_A_F4>G_V5_F4_REF else ""}'      f'${abs(G_A_F4-G_V5_F4_REF)/1e6:.1f}M')

# Train final: 4 meses completos + oversampling
mask_tf = df[COL_PERIODO].isin([202103,202104,202105,202106]) & df[TARGET].notna()
X_tf_a  = df.loc[mask_tf, FEATURE_COLS].copy()
y_tf_a  = df.loc[mask_tf, TARGET].astype(int).copy()
mask_p  = df[COL_PERIODO] == MES_PREDECIR
X_pred  = df.loc[mask_p, FEATURE_COLS].copy()
ids_p   = df.loc[mask_p, COL_ID].values
print(f'Train final: {len(y_tf_a):,} | 4 meses completos')

probs_a, _, ts_a = semillerio_y_submission(
    PARAMS_V5, X_tf_a, y_tf_a, X_pred, ids_p,
    N_OPT_A, RUTA_SUBS_A, 'v13a', CKPT_PROBS_13A, CKPT_IDS_13A,
    usar_oversampling=True, es_dart=ES_DART)
del X_tf_a, y_tf_a; gc.collect()
print(f'✓ EXP A completo')


EXP A — A1 + 4 MESES TRAIN FINAL (submissions10/)
  [A fold_2] AUC=0.9118 | Ganancia=$532,702,500 | N=9,349.0 | TP=718.0 | thresh=0.8410
  [A fold_3] AUC=0.8915 | Ganancia=$273,982,500 | N=7,637.0 | TP=440.0 | thresh=0.8950
    Oversampling ×2 p50: 1,493 baja2 duplicados
    Train: 486,826 → 488,319
  [A fold_4] AUC=0.9027 | Ganancia=$419,292,500 | N=15,953.0 | TP=780.0 | thresh=0.7050

fold_4: $419,292,500 | vs v5: $11.7M
Train final: 650,066 | 4 meses completos
    Oversampling ×2 p50: 2,045 baja2 duplicados
    Train: 650,066 → 652,111
  Semilla 42 (1/4)... ✓
  Semilla 123 (2/4)... ✓
  Semilla 777 (3/4)... ✓
  Semilla 2024 (4/4)... ✓
  20 archivos | sin notación científica: ✓
  Cortes: [15000, 15000, 15000, 15000, 15000]
  → submit v13a_1011_2337 | submissions10/
✓ EXP A completo


---
## 4 · Experimento B — A1 + ventana 4 meses en walk-forward

Acá la diferencia es que **el walk-forward de fold_4 también usa 4 meses**
(202103+04+05+06 como train, test sobre 202106 — lo que implica que 202106
aparece en train y test simultáneamente).

**Nota:** esto es un experimento especulativo. Fold_4 con 4 meses de train
incluye 202106 que es el mes de test — técnicamente hay leakage en el WF.
Pero el modelo final sí puede usar los 4 meses limpiamente porque predice 202108.
Lo usamos solo para estimar el N_OPT del semillerio.

In [ ]:
print('\n' + '='*60)
print('EXP B — A1 + VENTANA 4 MESES WF (submissions11/)')
print('='*60)

rows_b = []
for fold in FOLDS:
    # fold_4 usa todos los meses disponibles (incluyendo 202106)
    if fold['nombre'] == 'fold_4':
        meses_tr = [202103, 202104, 202105]  # NO incluir 202106 en train de WF
        # Explicación: usar 202106 en train de fold_4 y en test sería leakage directo
        # En cambio entrenamos igual que siempre y el "4 meses" aplica solo al modelo final
    else:
        meses_tr = get_meses_train(fold['nombre'], N_MESES_V5)

    Xtr, ytr, Xte, yte = get_split(meses_tr, fold['test'])
    Xtr_aug, ytr_aug = aplicar_oversampling(Xtr, ytr, verbose=(fold['nombre']=='fold_4'))
    m = fit_modelo(PARAMS_V5, Xtr_aug, ytr_aug, Xte, yte, ES_DART)
    prob = m.predict_proba(Xte)[:, 1]
    res  = evaluar_fold(yte.values, prob, f'B {fold["nombre"]}')
    rows_b.append({'fold': fold['nombre'], 'experimento': 'v13b_a1_wf4m', **res})
    del Xtr, ytr, Xtr_aug, ytr_aug, Xte, yte, m, prob; gc.collect()

guardar_metricas(rows_b)
G_B_F4  = next(r for r in rows_b if r['fold']=='fold_4')['ganancia']
N_OPT_B = int(next(r for r in rows_b if r['fold']=='fold_4')['n_intervenidos'])
print(f'\nfold_4: ${G_B_F4:,.0f} | vs v5: {"+ " if G_B_F4>G_V5_F4_REF else ""}'      f'${abs(G_B_F4-G_V5_F4_REF)/1e6:.1f}M')

# Train final: 4 meses completos (mismo que Exp A) — diferencia es solo el N_OPT
mask_tf = df[COL_PERIODO].isin([202103,202104,202105,202106]) & df[TARGET].notna()
X_tf_b  = df.loc[mask_tf, FEATURE_COLS].copy()
y_tf_b  = df.loc[mask_tf, TARGET].astype(int).copy()

probs_b, _, ts_b = semillerio_y_submission(
    PARAMS_V5, X_tf_b, y_tf_b, X_pred, ids_p,
    N_OPT_B, RUTA_SUBS_B, 'v13b', CKPT_PROBS_13B, CKPT_IDS_13B,
    usar_oversampling=True, es_dart=ES_DART)
del X_tf_b, y_tf_b; gc.collect()
print(f'✓ EXP B completo')


EXP B — A1 + VENTANA 4 MESES WF (submissions11/)
  [B fold_2] AUC=0.9118 | Ganancia=$532,702,500 | N=9,349.0 | TP=718.0 | thresh=0.8410
  [B fold_3] AUC=0.8915 | Ganancia=$273,982,500 | N=7,637.0 | TP=440.0 | thresh=0.8950
    Oversampling ×2 p50: 1,493 baja2 duplicados
    Train: 486,826 → 488,319
  [B fold_4] AUC=0.9027 | Ganancia=$419,292,500 | N=15,953.0 | TP=780.0 | thresh=0.7050

fold_4: $419,292,500 | vs v5: $11.7M
    Oversampling ×2 p50: 2,045 baja2 duplicados
    Train: 650,066 → 652,111
  Semilla 42 (1/4)... ✓
  Semilla 123 (2/4)... ✓
  Semilla 777 (3/4)... ✓
  Semilla 2024 (4/4)... ✓
  20 archivos | sin notación científica: ✓
  Cortes: [15000, 15000, 15000, 15000, 15000]
  → submit v13b_1012_0157 | submissions11/
✓ EXP B completo


---
## 5 · Experimento C — Stacking simple

Dos modelos con hiperparámetros ligeramente distintos. Sus probabilidades
se promedian con pesos: 60% modelo base (v5) + 40% modelo alternativo.

La diversidad viene de:
- `num_leaves` más alto en el alternativo (más profundo, captura interacciones complejas)
- `learning_rate` más bajo (más conservador, diferente región del espacio)
- `feature_fraction` menor (ve subconjuntos distintos de features)

El promedio ponderado reduce varianza sin perder la señal del mejor modelo.

In [ ]:
print('\n' + '='*60)
print('EXP C — STACKING 2 MODELOS (submissions12/)')
print('='*60)

rows_c = []
for fold in FOLDS:
    meses_tr = get_meses_train(fold['nombre'], N_MESES_V5)
    Xtr, ytr, Xte, yte = get_split(meses_tr, fold['test'])
    Xtr_aug, ytr_aug = aplicar_oversampling(Xtr, ytr, verbose=(fold['nombre']=='fold_4'))

    # Modelo base
    m1 = fit_modelo(PARAMS_V5, Xtr_aug, ytr_aug, Xte, yte, ES_DART)
    p1 = m1.predict_proba(Xte)[:, 1]
    del m1; gc.collect()

    # Modelo alternativo
    m2 = fit_modelo(PARAMS_V5_ALT, Xtr_aug, ytr_aug, Xte, yte, ES_DART)
    p2 = m2.predict_proba(Xte)[:, 1]
    del m2; gc.collect()

    # Promedio ponderado 60/40
    prob_stack = 0.6 * p1 + 0.4 * p2
    res = evaluar_fold(yte.values, prob_stack, f'C stack {fold["nombre"]}')

    # También reportar cada modelo solo para comparar
    opt1 = threshold_optimo(yte.values, p1)
    opt2 = threshold_optimo(yte.values, p2)
    if fold['nombre'] == 'fold_4':
        print(f'    M1 solo:  ${opt1["ganancia"]:,.0f}')
        print(f'    M2 solo:  ${opt2["ganancia"]:,.0f}')
        print(f'    Stack 60/40: ${res["ganancia"]:,.0f}')

    rows_c.append({'fold': fold['nombre'], 'experimento': 'v13c_stack', **res})
    del Xtr, ytr, Xtr_aug, ytr_aug, Xte, yte, p1, p2; gc.collect()

guardar_metricas(rows_c)
G_C_F4  = next(r for r in rows_c if r['fold']=='fold_4')['ganancia']
N_OPT_C = int(next(r for r in rows_c if r['fold']=='fold_4')['n_intervenidos'])
print(f'\nfold_4 stack: ${G_C_F4:,.0f} | vs v5: {"+ " if G_C_F4>G_V5_F4_REF else ""}'      f'${abs(G_C_F4-G_V5_F4_REF)/1e6:.1f}M')

# Semillerio: para cada semilla entrenamos M1+M2 y promediamos
mask_tf = df[COL_PERIODO].isin([202103,202104,202105,202106]) & df[TARGET].notna()
X_tf_c  = df.loc[mask_tf, FEATURE_COLS].copy()
y_tf_c  = df.loc[mask_tf, TARGET].astype(int).copy()

X_tf_c_aug, y_tf_c_aug = aplicar_oversampling(X_tf_c, y_tf_c, verbose=True)
del X_tf_c; gc.collect()

if CKPT_PROBS_13C.exists() and CKPT_IDS_13C.exists():
    probs_c = list(np.load(CKPT_PROBS_13C))
    ids_ck  = np.load(CKPT_IDS_13C)
    n_ya    = len(probs_c) if np.array_equal(ids_ck, ids_p) else 0
    if n_ya == 0: probs_c = []
    else: print(f'  Checkpoint C: {n_ya}/{len(SEMILLAS)} semillas')
else:
    probs_c = []; n_ya = 0

for i_s, sem in enumerate(SEMILLAS[n_ya:], start=n_ya):
    print(f'  C semilla {sem} ({i_s+1}/{len(SEMILLAS)}):', end=' ')
    # M1
    m1 = lgb.LGBMClassifier(**{**PARAMS_V5, 'random_state': sem})
    m1.fit(X_tf_c_aug, y_tf_c_aug, callbacks=[lgb.log_evaluation(-1)])
    p1 = m1.predict_proba(X_pred)[:, 1]
    del m1; gc.collect()
    print('M1✓', end=' ')
    # M2
    m2 = lgb.LGBMClassifier(**{**PARAMS_V5_ALT, 'random_state': sem + 100})
    m2.fit(X_tf_c_aug, y_tf_c_aug, callbacks=[lgb.log_evaluation(-1)])
    p2 = m2.predict_proba(X_pred)[:, 1]
    del m2; gc.collect()
    print('M2✓', end=' ')
    probs_c.append(0.6 * p1 + 0.4 * p2)
    del p1, p2
    np.save(CKPT_PROBS_13C, np.array(probs_c))
    np.save(CKPT_IDS_13C, ids_p)
    print('✓ guardado')

del X_tf_c_aug, y_tf_c_aug; gc.collect()

# Generar CSVs
paso     = max(100, int(N_OPT_C * 0.025))
cortes_c = [max(8000, min(15000, N_OPT_C + i * paso)) for i in range(-2, 3)]
ts_c     = datetime.now().strftime('%m%d_%H%M')
archivos_c = []
for i_s, sem in enumerate(SEMILLAS):
    for n_corte in cortes_c:
        ranking = np.argsort(probs_c[i_s])[::-1][:n_corte]
        ids_int = ids_p[ranking].astype(int)
        nombre  = f'v13c_stack_sem{sem}_top{n_corte}_{ts_c}'
        pd.Series(ids_int).to_csv(RUTA_SUBS_C / f'{nombre}.csv', index=False, header=False)
        archivos_c.append(nombre)
pd.DataFrame({'nombre': archivos_c}).to_csv(RUTA_SUBS_C / f'log_{ts_c}.csv', index=False)
tiene_e = pd.read_csv(RUTA_SUBS_C / f'{archivos_c[0]}.csv', header=None)[0]            .astype(str).str.lower().str.contains('e').any()
print(f'  {len(archivos_c)} archivos | sin notación científica: {"✓" if not tiene_e else "✗"}')
print(f'  → submit v13c_stack_{ts_c} | submissions12/')
print(f'✓ EXP C completo')


EXP C — STACKING 2 MODELOS (submissions12/)
  [C stack fold_2] AUC=0.9127 | Ganancia=$532,427,500 | N=9,439.0 | TP=720.0 | thresh=0.8390
  [C stack fold_3] AUC=0.8935 | Ganancia=$268,565,000 | N=8,114.0 | TP=447.0 | thresh=0.8770
    Oversampling ×2 p50: 1,493 baja2 duplicados
    Train: 486,826 → 488,319
  [C stack fold_4] AUC=0.9038 | Ganancia=$408,897,500 | N=15,891.0 | TP=769.0 | thresh=0.7010
    M1 solo:  $419,292,500
    M2 solo:  $393,387,500
    Stack 60/40: $408,897,500

fold_4 stack: $408,897,500 | vs v5: $22.1M
    Oversampling ×2 p50: 2,045 baja2 duplicados
    Train: 650,066 → 652,111
  C semilla 42 (1/4): M1✓ M2✓ ✓ guardado
  C semilla 123 (2/4): M1✓ M2✓ ✓ guardado
  C semilla 777 (3/4): M1✓ M2✓ ✓ guardado
  C semilla 2024 (4/4): M1✓ M2✓ ✓ guardado
  20 archivos | sin notación científica: ✓
  → submit v13c_stack_1012_0835 | submissions12/
✓ EXP C completo


## 6 · Resumen final

In [ ]:
del df; gc.collect()

print('\n' + '='*60)
print('RESUMEN FINAL — Notebook 13')
print('='*60)
print(f'\n{"Experimento":>30} | {"Ganancia fold_4":>18} | {"vs v5 (87.07M pub)":>18}')
print('─' * 72)

resultados_13 = [
    ('v5 base (mejor público)',  G_V5_F4_REF, '—'),
    ('A: A1 + 4m train final',   G_A_F4,      f'submissions10/ ({ts_a})'),
    ('B: A1 + N_OPT de 4m WF',  G_B_F4,      f'submissions11/ ({ts_b})'),
    ('C: Stack M1×0.6+M2×0.4',  G_C_F4,      f'submissions12/ ({ts_c})'),
]
for nombre, g, sub in resultados_13:
    delta = g - G_V5_F4_REF
    signo = '+' if delta > 0 else ''
    marca = ' ✓' if g > G_V5_F4_REF else ''
    print(f'{nombre:>30} | ${g:>15,.0f} | {signo}${delta/1e6:>8.1f}M{marca}')

ranking = sorted(
    [('A', G_A_F4, 'submissions10'), ('B', G_B_F4, 'submissions11'),
     ('C', G_C_F4, 'submissions12')],
    key=lambda x: -x[1])

print('\nPrioridad de submits:')
for i, (exp, g, sub) in enumerate(ranking, 1):
    supera = "✓ supera v5" if g > G_V5_F4_REF else "✗ no supera v5"
    print(f'  {i}. Exp {exp}: ${g:,.0f} ({supera}) → {sub}/')

print('\n✓ Notebook 13 completada.')


RESUMEN FINAL — Notebook 13

                   Experimento |    Ganancia fold_4 | vs v5 (87.07M pub)
────────────────────────────────────────────────────────────────────────
       v5 base (mejor público) | $    430,980,000 | $     0.0M
        A: A1 + 4m train final | $    419,292,500 | $   -11.7M
        B: A1 + N_OPT de 4m WF | $    419,292,500 | $   -11.7M
        C: Stack M1×0.6+M2×0.4 | $    408,897,500 | $   -22.1M

Prioridad de submits:
  1. Exp A: $419,292,500 (✗ no supera v5) → submissions10/
  2. Exp B: $419,292,500 (✗ no supera v5) → submissions11/
  3. Exp C: $408,897,500 (✗ no supera v5) → submissions12/

✓ Notebook 13 completada.


In [20]:
import requests
import base64
import os

GITHUB_TOKEN = "TU_TOKEN_AQUI"
REPO         = "marcostrigueros99/churn-bancario-dmeyf"
BRANCH       = "main"
RUTA_COMP    = '/content/drive/MyDrive/Maestría Data Mining/DM en EyF/Competencia 1'

# Verificar token primero
headers = {"Authorization": f"token {GITHUB_TOKEN}",
           "Accept": "application/vnd.github.v3+json"}
r = requests.get("https://api.github.com/user", headers=headers)
usuario = r.json().get('login')
if usuario:
    print(f'✓ Token válido — usuario: {usuario}')
else:
    print(f'✗ Token inválido: {r.json()}')
    raise Exception("Corregí el token antes de continuar")

def subir_archivo(ruta_local, ruta_repo):
    with open(ruta_local, 'rb') as f:
        contenido = base64.b64encode(f.read()).decode('utf-8')
    url = f"https://api.github.com/repos/{REPO}/contents/{ruta_repo}"
    r   = requests.get(url, headers=headers)
    sha = r.json().get('sha') if r.status_code == 200 else None
    payload = {"message": f"Add/update {os.path.basename(ruta_repo)}",
               "content": contenido, "branch": BRANCH}
    if sha:
        payload["sha"] = sha
    r = requests.put(url, headers=headers, json=payload)
    if r.status_code in [200, 201]:
        print(f"  ✓ {os.path.basename(ruta_repo)}")
    else:
        print(f"  ✗ {os.path.basename(ruta_repo)}: {r.json().get('message')}")

notebooks = [
    '01_eda_churn_bancario.ipynb',
    '02_calidad_datos_churn.ipynb',
    '03_construccion_target.ipynb',
    '04_fe_modelo_optuna_v3.ipynb',
    '05_submission_semillerio.ipynb',
    '06_modelo_v4_submision.ipynb',
    '07_experimentos_v5.ipynb',
    '08_experimentos_v5_bugfix.ipynb',
    '08b_sinleak.ipynb',
    '09_oversampling_fn.ipynb',
    '10_cascada.ipynb',
    '11_nocturnos.ipynb',
    '12_fe_v4.ipynb',
    '13_combinaciones.ipynb',
]

params = [
    'mejores_params_optuna_v3.json',
    'mejores_params_optuna_v4.json',
    'mejores_params_optuna_v5.json',
    'mejores_params_optuna_v10.json',
    'metricas_experimentos.csv',
]

print('\nSubiendo notebooks...')
for nb in notebooks:
    ruta_local = f'{RUTA_COMP}/{nb}'
    if os.path.exists(ruta_local):
        subir_archivo(ruta_local, f'notebooks/{nb}')
    else:
        print(f'  ✗ No encontrado: {nb}')

print('\nSubiendo params y métricas...')
for p in params:
    ruta_local = f'{RUTA_COMP}/{p}'
    if os.path.exists(ruta_local):
        carpeta = 'params' if p.endswith('.json') else 'metricas'
        subir_archivo(ruta_local, f'{carpeta}/{p}')
    else:
        print(f'  ✗ No encontrado: {p}')

print('\n✓ Subida completa.')

✓ Token válido — usuario: marcostrigueros99

Subiendo notebooks...
  ✓ 01_eda_churn_bancario.ipynb
  ✓ 02_calidad_datos_churn.ipynb
  ✓ 03_construccion_target.ipynb
  ✓ 04_fe_modelo_optuna_v3.ipynb
  ✓ 05_submission_semillerio.ipynb
  ✓ 06_modelo_v4_submision.ipynb
  ✓ 07_experimentos_v5.ipynb
  ✓ 08_experimentos_v5_bugfix.ipynb
  ✓ 08b_sinleak.ipynb
  ✓ 09_oversampling_fn.ipynb
  ✓ 10_cascada.ipynb
  ✓ 11_nocturnos.ipynb
  ✓ 12_fe_v4.ipynb
  ✗ 13_combinaciones.ipynb: Repository rule violations found

Secret detected in content



Subiendo params y métricas...
  ✓ mejores_params_optuna_v3.json
  ✓ mejores_params_optuna_v4.json
  ✓ mejores_params_optuna_v5.json
  ✓ mejores_params_optuna_v10.json
  ✓ metricas_experimentos.csv

✓ Subida completa.
